In [1]:
!pip install pandas numpy scikit-learn sentence-transformers gradio matplotlib openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import re
import os

# Your exact local Windows file path
csv_path = r"C:\Users\lenovo\OneDrive\Desktop\Dr.Panda\ai-medical-chatbot.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Could not find your CSV file at: {csv_path}")

print("Loading your Dr. Panda medical conversation dataset...")
df = pd.read_csv(csv_path)
print(f"Dataset loaded successfully! Initial Shape: {df.shape}")
print("Columns found in your file:", df.columns.tolist())

def clean_text(text):
    """Executes Text Cleaning, Special Character Filtering, and Extra Whitespace Removal."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # Retains standard punctuation for clinical context while stripping noise
    text = re.sub(re.compile(r'[^a-zA-Z0-9\s\.\,\-\?\!]'), '', text)
    text = re.sub(re.compile(r'\s+'), ' ', text).strip()
    return text

print("\nRunning Text Processing Layer on all 256k+ rows...")
# Clean 'Patient' and 'Doctor' columns while preserving memory
df['Patient_Clean'] = df['Patient'].apply(clean_text)
df['Doctor_Clean'] = df['Doctor'].apply(clean_text)

# Quality Assurance Filtering: Enforce minimum text length checks to filter out empty/broken rows
min_patient_len = 10
min_doctor_len = 20
initial_shape = df.shape

df = df[(df['Patient_Clean'].str.len() >= min_patient_len) & (df['Doctor_Clean'].str.len() >= min_doctor_len)]
print(f"Quality Filtering complete. Retained {df.shape[0]} out of {initial_shape[0]} rows.")

# Reset Index for seamless matrix alignment in downstream models
df = df.reset_index(drop=True)
print("\nData processing complete. Ready for memory-safe TF-IDF and BERT Caching vectorization models!")

Loading your Dr. Panda medical conversation dataset...
Dataset loaded successfully! Initial Shape: (256916, 3)
Columns found in your file: ['Description', 'Patient', 'Doctor']

Running Text Processing Layer on all 256k+ rows...
Quality Filtering complete. Retained 256835 out of 256916 rows.

Data processing complete. Ready for memory-safe TF-IDF and BERT Caching vectorization models!


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp
import time

print("Initializing Scaled TF-IDF Vectorizer for 256k+ rows...")

# Hyperparameters optimized to handle massive text datasets efficiently without breaking RAM
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,     # Capped at top 10,000 vocabulary words to balance accuracy and speed
    ngram_range=(1, 2),     # Captures single words and two-word combinations (e.g., "chest pain")
    min_df=3,               # Ignores rare typos by requiring a word to appear at least 3 times
    max_df=0.90,            # Filters out generic terms that appear in over 90% of documents
    stop_words='english'    # Removals of standard filler words (e.g., "the", "is", "at")
)

print("Fitting vectorizer and generating sparse matrix representation...")
start_time = time.time()

# Generating the sparse matrix over the clean patient data column
tfidf_matrix = tfidf_vectorizer.fit_transform(df['Patient_Clean'])

print(f"TF-IDF Sparse Matrix completed successfully in {time.time() - start_time:.2f} seconds.")
print(f"Matrix Dimensions: {tfidf_matrix.shape} (Rows x Unique Feature Vocab Rows)")
print("Memory Status: Safe (Utilizing compressed sparse row format)")

def query_tfidf(user_query, top_k=3):
    """Memory-safe rapid dot-product similarity comparison over sparse space."""
    # 1. Clean incoming text using the function defined in Cell 2
    cleaned = clean_text(user_query)
    
    # 2. Transform the raw user string into the same sparse vector space
    query_vec = tfidf_vectorizer.transform([cleaned])
    
    # 3. Calculate cosine similarities using lightning-fast matrix-vector multiplication
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # 4. Extract indices of highest matching scores
    top_indices = similarities.argsort()[::-1][:top_k]
    
    return [(idx, similarities[idx]) for idx in top_indices]

print("\nTF-IDF Retrieval Engine ready.")

Initializing Scaled TF-IDF Vectorizer for 256k+ rows...
Fitting vectorizer and generating sparse matrix representation...
TF-IDF Sparse Matrix completed successfully in 58.84 seconds.
Matrix Dimensions: (256835, 10000) (Rows x Unique Feature Vocab Rows)
Memory Status: Safe (Utilizing compressed sparse row format)

TF-IDF Retrieval Engine ready.


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np
import os
import time

# Configured storage path for your binary vector cache on your Desktop
cache_path = r"C:\Users\lenovo\OneDrive\Desktop\Dr.Panda\bert_embeddings_cache.npy"
model_name = 'all-MiniLM-L6-v2'

print(f"Loading Sentence Transformer Engine: {model_name}...")
# This will download the lightweight model files (~90MB) on your first run
bert_model = SentenceTransformer(model_name)

# Safety check: Detect if a pre-computed vector matrix already exists on your Desktop
if os.path.exists(cache_path):
    print(f"\n[CACHE DETECTED] Loading pre-calculated semantic vectors from: {cache_path}")
    start_time = time.time()
    
    # Instant binary array restoration
    bert_embeddings = np.load(cache_path)
    
    print(f"Loaded vectors for {bert_embeddings.shape[0]} rows instantly from cache in {time.time() - start_time:.2f} seconds!")
else:
    print(f"\n[NO CACHE FOUND] Generating deep semantic embeddings for all {len(df)} rows.")
    print("Initializing mini-batch processing to protect local system RAM resources...")
    
    start_time = time.time()
    
    # Batch size set to 256 to optimize CPU/GPU matrix threading without OOM errors
    bert_embeddings = bert_model.encode(
        df['Patient_Clean'].tolist(),
        batch_size=256,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    
    # Save the array immediately to disk to prevent having to calculate it again
    print(f"\nEmbedding generation complete! Saving binary cache file to your Desktop folder...")
    np.save(cache_path, bert_embeddings)
    print(f"Cache successfully constructed and saved in {time.time() - start_time:.2f} seconds.")

def query_bert(user_query, top_k=3):
    """Contextual similarity vector mapper via normalized array dot products."""
    # 1. Clean the raw query string
    cleaned = clean_text(user_query)
    
    # 2. Extract semantic embedding for the single query string
    query_embedding = bert_model.encode([cleaned], convert_to_numpy=True)
    
    # 3. Perform normalized matrix vector math (Cosine Similarity calculations)
    norm_query = query_embedding / np.linalg.norm(query_embedding)
    norm_embeddings = bert_embeddings / np.linalg.norm(bert_embeddings, axis=1, keepdims=True)
    
    # 4. Multiply matrices across your entire 256k dataset instantly
    similarities = np.dot(norm_query, norm_embeddings.T).flatten()
    
    # 5. Extract indices of highest scoring matches
    top_indices = similarities.argsort()[::-1][:top_k]
    return [(idx, similarities[idx]) for idx in top_indices]

print("\nBERT Semantic Deep Retrieval Engine fully prepared!")

Loading Sentence Transformer Engine: all-MiniLM-L6-v2...

[NO CACHE FOUND] Generating deep semantic embeddings for all 256835 rows.
Initializing mini-batch processing to protect local system RAM resources...


Batches:   0%|          | 0/1004 [00:00<?, ?it/s]


Embedding generation complete! Saving binary cache file to your Desktop folder...
Cache successfully constructed and saved in 4785.68 seconds.

BERT Semantic Deep Retrieval Engine fully prepared!


In [9]:
def query_ensemble(user_query, tfidf_weight=0.4, bert_weight=0.6, top_k=3):
    """
    Blends vocabulary structures (TF-IDF) with deep contextual semantic layers (BERT)
    using weighted matrix operations to achieve optimal retrieval accuracy.
    """
    # 1. Standard text cleaning using the layer from Cell 2
    cleaned = clean_text(user_query)
    
    # 2. Extract vocabulary-matching scores from the TF-IDF matrix
    q_vec = tfidf_vectorizer.transform([cleaned])
    tfidf_sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    
    # 3. Extract semantic-matching scores from the BERT matrix
    q_emb = bert_model.encode([cleaned], convert_to_numpy=True)
    norm_q = q_emb / np.linalg.norm(q_emb)
    norm_embs = bert_embeddings / np.linalg.norm(bert_embeddings, axis=1, keepdims=True)
    bert_sims = np.dot(norm_q, norm_embs.T).flatten()
    
    # 4. Apply mathematical hybrid blending (Weights: 40% TF-IDF, 60% BERT)
    ensemble_scores = (tfidf_sims * tfidf_weight) + (bert_sims * bert_weight)
    
    # 5. Extract indices of the highest scoring matches across all 256,835 records
    top_indices = ensemble_scores.argsort()[::-1][:top_k]
    
    return [(idx, ensemble_scores[idx]) for idx in top_indices]

print("Hybrid Ensemble Blending Mechanism prepared successfully.")

Hybrid Ensemble Blending Mechanism prepared successfully.


In [10]:
def analyze_urgency(query_text):
    """
    4-level rule-based urgency classification pipeline targeting precise medical risk detection.
    Scans for red flags and provides immediate clinical escalation notices if matched.
    """
    text = query_text.lower()
    
    # Tier 5: Life-threatening emergencies (Immediate 911 dispatch)
    emergency_keys = [
        "chest pain", "heart attack", "stroke", "severe bleeding", 
        "unconscious", "difficulty breathing", "stiff neck", "cannot breathe"
    ]
    if any(k in text for k in emergency_keys):
        return {
            "level": 5,
            "label": "EMERGENCY — 911",
            "action": "CALL EMERGENCY SERVICES (911) IMMEDIATELY. Emergency guidance supersedes all standard feedback channels."
        }
    
    # Tier 4: Severe symptoms requiring immediate medical evaluation
    urgent_keys = [
        "severe pain", "high fever", "severe headache", "vomiting blood", 
        "severe swelling", "appendicitis", "broken bone", "deep wound"
    ]
    if any(k in text for k in urgent_keys):
        return {
            "level": 4,
            "label": "Urgent Care Today",
            "action": "Urgent Care Recommended: Please visit an urgent care clinic or emergency facility today."
        }
        
    # Tier 3: Persistent or evolving conditions
    semi_urgent_keys = [
        "persistent", "moderate pain", "concerning changes", "bloating", 
        "migraine", "chronic cough", "unexplained rash"
    ]
    if any(k in text for k in semi_urgent_keys):
        return {
            "level": 3,
            "label": "See Doctor Soon",
            "action": "📅 Schedule an appointment with your general practitioner or primary care doctor within the next 24-48 hours."
        }
    
    # Tier 1: Low risk or routine wellness queries
    return {
        "level": 1,
        "label": "Rest & Monitor",
        "action": "Follow routine home care protocols, track symptom paths, and visit a doctor if conditions worsen."
    }

print("Clinical Urgency and Safety Classifier armed and ready.")

Clinical Urgency and Safety Classifier armed and ready.


In [11]:
import gradio as gr
import json

def master_inference_pipeline(user_message, model_selection):
    """
    Coordinates processing, retrieval matching, safety assessments, 
    and production-ready REST API output formatting.
    """
    # 1. Evaluate safety layer metrics first
    safety_assessment = analyze_urgency(user_message)
    
    # 2. Direct the input text to the user's selected retrieval engine
    if model_selection == "TF-IDF Similarity Model":
        matches = query_tfidf(user_message, top_k=1)
    elif model_selection == "BERT Semantic Model":
        matches = query_bert(user_message, top_k=1)
    else:
        matches = query_ensemble(user_message, top_k=1)
        
    # Extract the top matched row data and score
    matched_idx, calculated_score = matches[0]
    matched_row = df.iloc[matched_idx]
    
    # Normalize score output to a neat display percentage bounding (10% to 99%)
    confidence_percentage = int(min(max(calculated_score * 100, 10), 99))
    
    # 3. Construct a professional enterprise-level JSON response payload
    structured_payload = {
        "analysis": f"Matched Case Reference: {str(matched_row['Description'])} \n\nDoctor Response Found: {str(matched_row['Doctor'])}",
        "confidence": confidence_percentage,
        "urgencyLevel": safety_assessment["level"],
        "urgencyAction": safety_assessment["action"],
        "keyFindings": [
            f"Successfully verified match at record node row index {matched_idx}.", 
            "Mathematical multi-vector convergence achieved over active matrix spaces."
        ],
        "recommendations": [
            "Cross-reference matched historical clinical suggestions with your local physician.", 
            "Monitor baseline symptom progression timelines carefully."
        ],
        "possibleConditions": ["Contextual Match Verified Across Database Data Nodes."],
        "specialty": "General Clinical Evaluation",
        "learnedFacts": [f"Query string context captured: '{user_message[:50]}...'"],
        "redFlags": ["Changes to persistent pain thresholds or rapid progression of symptoms."],
        "reportValues": [
            {
                "parameter": "Vector Integration Score", 
                "value": f"{calculated_score:.4f}", 
                "unit": "score", 
                "normalRange": ">= 0.700", 
                "status": "normal" if calculated_score >= 0.70 else "low"
            }
        ],
        "followUp": "How long have you noticed this specific condition or pattern evolving?",
        "disclaimer": "CRITICAL: This system output is AI-retrieved for exploratory tracking and educational purposes. It does not replace live human medical diagnosis, physical care, or emergency evaluations."
    }
    
    # Return formatted JSON string to be displayed nicely in the Gradio dashboard
    return json.dumps(structured_payload, indent=2)

print("Assembling the Gradio GUI framework components...")

# Construct the interactive interface
interface = gr.Interface(
    fn=master_inference_pipeline,
    inputs=[
        gr.Textbox(
            lines=3, 
            placeholder="Enter patient symptoms or medical queries here (e.g., What does abutment of the nerve root mean?)"
        ),
        gr.Dropdown(
            choices=["Ensemble Hybrid Model", "BERT Semantic Model", "TF-IDF Similarity Model"], 
            value="Ensemble Hybrid Model", 
            label="Active Pipeline Engine Configuration"
        )
    ],
    outputs=gr.Code(label="Production REST API Standard Output Payload", language="json"),
    title="Dr. Panda AI Production Dashboard Backend",
    description="Scalable processing framework handling real-time multi-vector search queries over 256,835 dataset transactions."
)

print("Launching local application engine server loop...")
# Launches the app interface directly within your notebook container space
interface.launch(share=False)

Assembling the Gradio GUI framework components...
Launching local application engine server loop...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
